<a href="https://colab.research.google.com/github/ivanduzunov/AI-Agents-and-Workflows-for-Developers/blob/main/final_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# festival planner
!pip install q langchain langchain-openai

In [67]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.types import Send, interrupt, Command
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from typing import TypedDict, Literal, List, Annotated, Dict, Optional
from IPython.display import Image
from google.colab import userdata
from pydantic import BaseModel

openai_key = userdata.get('OPENAI_KEY')
llm = ChatOpenAI(
    model="gpt-5-nano",
    api_key=openai_key
)

festival_llm = llm.with_structured_output(FestivalResults)

In [ ]:
# research node - find festival
#

In [68]:
class ConcertPlannerState(TypedDict):
    user_request: str

    festivals: list
    selected_festival: Optional[dict]

    accommodation_type: Optional[str]
    accommodations: list
    selected_accommodation: Optional[dict]

    flights: list
    selected_flight: Optional[dict]

    purchase_approved: bool

    final_result: Optional[str]

class Festival(BaseModel):
    name: str
    country: str
    city: str
    price: float
    genre: str
    dates: str

class FestivalResults(BaseModel):
    festivals: list[Festival]

In [69]:
def research(state: ConcertPlannerState):

    user_request = state["user_request"]

    result = festival_llm.invoke(
        f"""
        Find suitable music festivals in Europe based on
        the following user request:

        {user_request}

        Return several suitable festivals.
        """
    )

    festivals = [
        festival.model_dump()
        for festival in result.festivals
    ]

    return {
        "festivals": festivals
    }

def select_festival(state: ConcertPlannerState):

    festivals = state["festivals"]

    print("TYPE:", type(festivals))
    print("FESTIVALS:", festivals)

    selection = interrupt({
        "type": "festival_selection",
        "message": "Choose a festival:",
        "options": festivals
    })

    return {
        "selected_festival": festivals[selection]
    }

def select_accommodation_type(state: ConcertPlannerState):

    options = ["hotel", "camping"]

    selection = interrupt({
        "type": "accommodation_type",
        "message": "Where would you like to stay?",
        "options": options
    })

    return {
        "accommodation_type": options[selection]
    }

def find_accommodation(state: ConcertPlannerState):

    festival = state["selected_festival"]
    accommodation_type = state["accommodation_type"]

    if accommodation_type == "camping":

        camping = {
            "name": f"{festival['name']} Official Camping",
            "type": "camping",
            "city": festival["city"],
            "price": 0,
            "description": "Official festival camping."
        }

        print(
            f"\n🏕️ Automatically selected: {camping['name']}"
        )

        return {
            "accommodations": [camping],
            "selected_accommodation": camping
        }

    result = accommodation_llm.invoke(
        f"""
        You are an accommodation research assistant.

        Find suitable hotels near this music festival:

        Festival:
        {festival}

        Find several suitable hotels.

        Consider:
        - proximity to the festival
        - price
        - location
        - value for money

        Return several suitable options.
        """
    )

    accommodations = [
        accommodation.model_dump()
        for accommodation in result.accommodations
    ]

    print(
        f"\n🏨 Found {len(accommodations)} hotels "
        f"near {festival['city']}"
    )

    return {
        "accommodations": accommodations
    }

def accommodation_router(state: ConcertPlannerState):

    if state["accommodation_type"] == "camping":
        return "travel"

    return "select"

def select_accommodation(state: ConcertPlannerState):

    accommodations = state["accommodations"]

    selection = interrupt({
        "type": "accommodation_selection",
        "message": "Choose accommodation",
        "options": accommodations
    })

    selected = accommodations[selection]

    print(f"\n🏨 Selected: {selected['name']}")

    return {
        "selected_accommodation": selected
    }

    # accommodations = state["accommodations"]

    # user_selection = interrupt({
    #     "type": "accommodation_selection",
    #     "message": "Choose your accommodation:",
    #     "options": accommodations
    # })

    # selected_accommodation = accommodations[user_selection]

    # return {
    #     "selected_accommodation": selected_accommodation
    # }

def research_travel(state: ConcertPlannerState):

    festival = state["selected_festival"]

    print(
        f"\n✈️ Searching flights to {festival['city']}..."
    )

    flights = [
        {
            "airline": "Airline A",
            "route": f"Sofia → {festival['city']}",
            "price": 120
        },
        {
            "airline": "Airline B",
            "route": f"Sofia → {festival['city']}",
            "price": 160
        },
        {
            "airline": "Airline C",
            "route": f"Sofia → {festival['city']}",
            "price": 200
        }
    ]

    return {
        "flights": flights
    }

    # festival = state["selected_festival"]
    # accommodation = state["selected_accommodation"]

    # flights = travel_agent.invoke({
    #     "festival": festival,
    #     "accommodation": accommodation
    # })

    # return {
    #     "flights": flights
    # }

def select_flight(state: ConcertPlannerState):

    flights = state["flights"]

    user_selection = interrupt({
        "type": "flight_selection",
        "message": "Choose your flight:",
        "options": flights
    })

    selected_flight = flights[user_selection]

    return {
        "selected_flight": selected_flight
    }

def confirm_purchase(state: ConcertPlannerState):

    festival = state["selected_festival"]
    accommodation = state["selected_accommodation"]
    flight = state["selected_flight"]

    total_price = (
        festival["price"]
        + accommodation["price"]
        + flight["price"]
    )

    approval = interrupt({
        "type": "purchase_confirmation",
        "message": "Please review your trip before purchase.",
        "festival": festival,
        "accommodation": accommodation,
        "flight": flight,
        "total_price": total_price
    })

    return {
        "purchase_approved": approval
    }

def buy_tickets(state: ConcertPlannerState):

    if not state["purchase_approved"]:
        return {
            "final_result": "Purchase cancelled by user."
        }

    print("\n PURCHASING...")

    return {
        "final_result": (
            "Festival ticket purchased.\n"
            "Accommodation booked.\n"
            "Flight booked."
        )
    }

In [71]:
# graph_builder = StateGraph(ConcertPlannerState)
# graph_builder.add_node("Research", empty_fn)
# graph_builder.add_node("Find accomodation", empty_fn)
# graph_builder.add_node("Research travel", empty_fn)
# graph_builder.add_node("Buy tickets", empty_fn)

# graph_builder.add_edge(START, "Research")
# graph_builder.add_edge("Research", "Find accomodation")
# graph_builder.add_edge("Find accomodation", "Research travel")
# graph_builder.add_edge("Research travel", "Buy tickets")


# graph = graph_builder.compile(debug=True)

graph_builder = StateGraph(ConcertPlannerState)
graph_builder.add_node("Research", research)
graph_builder.add_node("Select Festival", select_festival)
graph_builder.add_node("Select Accommodation Type", select_accommodation_type)
graph_builder.add_node("Find Accommodation", find_accommodation)
graph_builder.add_node("Select Accommodation", select_accommodation)
graph_builder.add_node("Research Travel", research_travel)
graph_builder.add_node("Select Flight", select_flight)
graph_builder.add_node("Confirm Purchase", confirm_purchase)
graph_builder.add_node("Buy Tickets", buy_tickets)

graph_builder.add_edge(START, "Research")
graph_builder.add_edge("Research", "Select Festival")
graph_builder.add_edge("Select Festival", "Select Accommodation Type")
graph_builder.add_edge("Select Accommodation Type", "Find Accommodation")
graph_builder.add_conditional_edges(
    "Find Accommodation",
    accommodation_router,
    {
        "select": "Select Accommodation",
        "travel": "Research Travel"
    }
)
graph_builder.add_edge("Select Accommodation", "Research Travel")
graph_builder.add_edge("Research Travel", "Select Flight")
graph_builder.add_edge("Select Flight", "Confirm Purchase")
graph_builder.add_edge("Confirm Purchase", "Buy Tickets")
graph_builder.add_edge("Buy Tickets", END)

checkpointer = InMemorySaver()

graph = graph_builder.compile(debug=True, checkpointer=checkpointer)

In [72]:
def execute_workflow(user_request: str):

    config = {
        "configurable": {
            "thread_id": "concert-planner-1"
        }
    }

    initial_state = {
        "user_request": user_request,
        "festivals": [],
        "selected_festival": None,
        "accommodation_type": None,
        "accommodations": [],
        "selected_accommodation": None,
        "flights": [],
        "selected_flight": None,
        "purchase_approved": False,
        "final_result": None
    }

    result = graph.invoke(
        initial_state,
        config=config
    )

    while "__interrupt__" in result:

        interrupt_value = result["__interrupt__"][0].value

        print("\n" + "=" * 50)
        print(interrupt_value["message"])
        print("=" * 50)

        options = interrupt_value["options"]

        for i, option in enumerate(options):
            print(f"\n[{i}]")

            if isinstance(option, dict):
                for key, value in option.items():
                    print(f"{key}: {value}")
            else:
                print(option)

        selection = int(
            input("\nYour choice: ")
        )

        result = graph.invoke(
            Command(resume=selection),
            config=config
        )

    return result

In [74]:
execute_workflow(
    "I want a rock festival in Europe during summer."
)

[values] {'user_request': 'I want a rock festival in Europe during summer.', 'festivals': [], 'selected_festival': None, 'accommodation_type': None, 'accommodations': [], 'selected_accommodation': None, 'flights': [], 'selected_flight': None, 'purchase_approved': False, 'final_result': None}
[updates] {'Research': {'festivals': [{'name': 'Rock am Ring', 'country': 'Germany', 'city': 'Nürburgring', 'price': 160.0, 'genre': 'Rock/Hard Rock', 'dates': 'June (summer)'}, {'name': 'Download Festival', 'country': 'United Kingdom', 'city': 'Castle Donington', 'price': 250.0, 'genre': 'Rock/Metal', 'dates': 'June (summer)'}, {'name': 'Glastonbury Festival', 'country': 'United Kingdom', 'city': 'Pilton, Somerset', 'price': 300.0, 'genre': 'Rock/Alternative', 'dates': 'June (summer)'}, {'name': 'Primavera Sound', 'country': 'Spain', 'city': 'Barcelona', 'price': 180.0, 'genre': 'Rock/Indie/Alternative', 'dates': 'June (summer)'}, {'name': 'Hellfest Open Air', 'country': 'France', 'city': 'Clisson

NameError: name 'accommodation_llm' is not defined

In [ ]:
display(Image(graph.get_graph(xray=True).draw_mermaid_png(), format="png"))

In [39]:
config = {
    "configurable": {
        "thread_id": "concert-demo-1"
    }
}

initial_state = {
    "user_request": (
        "I want a rock festival in Europe "
        "during the summer."
    ),

    "festivals": [],
    "selected_festival": None,

    "accommodation_type": None,
    "accommodations": [],
    "selected_accommodation": None,

    "flights": [],
    "selected_flight": None,

    "purchase_approved": False,
    "final_result": None
}


result = graph.invoke(
    initial_state,
    config=config
)

print(result)

[values] {'user_request': 'I want a rock festival in Europe during the summer.', 'festivals': [], 'selected_festival': None, 'accommodation_type': None, 'accommodations': [], 'selected_accommodation': None, 'flights': [], 'selected_flight': None, 'purchase_approved': False, 'final_result': None}

🔎 RESEARCH
Searching for festivals based on:
I want a rock festival in Europe during the summer.
[updates] {'Research': {'festivals': [{'name': 'Rock Festival Germany', 'country': 'Germany', 'city': 'Berlin', 'price': 180}, {'name': 'Rock Festival Austria', 'country': 'Austria', 'city': 'Vienna', 'price': 220}, {'name': 'Rock Festival Czechia', 'country': 'Czechia', 'city': 'Prague', 'price': 150}]}}
[values] {'user_request': 'I want a rock festival in Europe during the summer.', 'festivals': [{'name': 'Rock Festival Germany', 'country': 'Germany', 'city': 'Berlin', 'price': 180}, {'name': 'Rock Festival Austria', 'country': 'Austria', 'city': 'Vienna', 'price': 220}, {'name': 'Rock Festival C